# Section 4 Onboarding Notebook: Restricted Hartree–Fock (RHF) SCF & Molecular Properties from Scratch

### Overview & Pedagogical Design
In this notebook, **no Hartree–Fock or property analysis functions are pre-solved for you**. You will implement every mathematical step of a Restricted Hartree–Fock (RHF) self-consistent field (SCF) program and post-SCF property analyzer across **16 granular `# TODO` functions**:

1. **Part 1 — Extracting Molecular Integrals with [PySCF](https://pyscf.org/user/gto.html) (TODOs 1–2)**:
   - `extract_ao_integrals_from_pyscf`: Build a `pyscf.gto.Mole` object and extract $\mathbf{S}$ (`int1e_ovlp`), $\mathbf{T}$ (`int1e_kin`), $\mathbf{V}$ (`int1e_nuc`), $\mathbf{H}^{\text{core}} = \mathbf{T} + \mathbf{V}$, 4-index ERIs $(\mu\nu|\lambda\sigma)$ (`int2e`), dipole integrals $\langle\chi_\mu|\hat{\mathbf{r}}|\chi_\nu\rangle$ (`int1e_r`), and $E_{\text{nuc}}$ ([PySCF AO Integrals](https://github.com/pyscf/pyscf-doc/blob/master/examples/gto/20-ao_integrals.py)).
   - `verify_eri_8fold_symmetry`: Verify the 8-fold permutational symmetry of real-valued 4-index AO ERIs $(\mu\nu|\lambda\sigma) = (\nu\mu|\lambda\sigma) = (\mu\nu|\sigma\lambda) = (\lambda\sigma|\mu\nu)$.
2. **Part 2 — Symmetric Orthogonalization & Initial Core Guess (TODOs 3–5)**:
   - `build_symmetric_orthogonalizer`: Diagonalize $\mathbf{S} = \mathbf{U}\mathbf{s}\mathbf{U}^\mathsf{T}$ to form $\mathbf{X} = \mathbf{S}^{-1/2} = \mathbf{U}\mathbf{s}^{-1/2}\mathbf{U}^\mathsf{T}$ and verify $\mathbf{X}^\mathsf{T}\mathbf{S}\mathbf{X} = \mathbf{I}$ ([Crawford Project 03](https://pycrawfordprogproj.readthedocs.io/en/latest/Project_03/Project_03.html)).
   - `solve_roothaan_hall_step`: Transform any AO Fock matrix $\mathbf{F}$ into the orthogonalized AO basis ($\mathbf{F}' = \mathbf{X}^\mathsf{T}\mathbf{F}\mathbf{X}$), diagonalize $\mathbf{F}'\mathbf{C}' = \mathbf{C}'\boldsymbol{\varepsilon}$, and back-transform $\mathbf{C} = \mathbf{X}\mathbf{C}'$.
   - `compute_density_matrix`: Slice the lowest $n_{\text{occ}} = N_{\text{elec}}/2$ occupied columns of $\mathbf{C}$ and contract $D_{\mu\nu} = \sum_{i=1}^{n_{\text{occ}}} C_{\mu i}C_{\nu i}$.
3. **Part 3 — Forming Coulomb ($\mathbf{J}$), Exchange ($\mathbf{K}$), Fock ($\mathbf{F}$), and RHF Energy (TODOs 6–8)**:
   - `compute_coulomb_and_exchange_matrices`: Contract $\mathbf{D}$ with $(\mu\nu|\lambda\sigma)$ via `np.einsum` to form $J_{\mu\nu} = \sum_{\lambda\sigma}(\mu\nu|\lambda\sigma)D_{\lambda\sigma}$ and $K_{\mu\nu} = \sum_{\lambda\sigma}(\mu\lambda|\nu\sigma)D_{\lambda\sigma}$.
   - `build_fock_matrix`: Form $\mathbf{F} = \mathbf{H}^{\text{core}} + 2\mathbf{J} - \mathbf{K}$.
   - `compute_rhf_energies`: Evaluate $E_{\text{elec}} = \sum_{\mu\nu} D_{\mu\nu}(H^{\text{core}}_{\mu\nu} + F_{\mu\nu})$ and $E_{\text{total}} = E_{\text{elec}} + E_{\text{nuc}}$.
4. **Part 4 — Pulay DIIS Extrapolation & Complete SCF Loop (TODOs 9–11)**:
   - `compute_pulay_orbital_gradient`: Compute the orthonormalized commutator residual $\mathbf{e}_k = \mathbf{X}^\mathsf{T}(\mathbf{F}_k\mathbf{D}_k\mathbf{S} - \mathbf{S}\mathbf{D}_k\mathbf{F}_k)\mathbf{X}$ ([Psi4NumPy RHF-DIIS](https://github.com/psi4/psi4numpy/blob/master/Tutorials/03_Hartree-Fock/3b_rhf-diis.ipynb)).
   - `solve_diis_extrapolation`: Construct the $(m+1)\times(m+1)$ Pulay $\mathbf{B}$-matrix ($B_{ij} = \text{Tr}(\mathbf{e}_i^\mathsf{T}\mathbf{e}_j)$), solve for weights $\{c_i\}$, and return $\mathbf{F}_{\text{DIIS}} = \sum_{i=1}^m c_i\mathbf{F}_i$.
   - `run_rhf_scf_loop`: Combine TODOs 3–10 into a self-consistent field loop until $|\Delta E_k| < e_{\text{tol}}$ and $\|\mathbf{e}_k\|_{\text{RMS}} < d_{\text{tol}}$.
5. **Part 5 — Checking Energies, Wavefunction Invariants & Post-SCF Properties (TODOs 12–16)**:
   - `compute_energy_components_and_virial`: Decompose $E_{\text{tot}}$ into $E_{\text{kin}}, E_{\text{ne}}, E_J, E_K, E_{\text{nuc}}$ and evaluate the virial ratio $-\langle\hat{V}\rangle/\langle\hat{T}\rangle$.
   - `verify_wavefunction_invariants`: Check MO orthonormality ($\mathbf{C}^\mathsf{T}\mathbf{S}\mathbf{C}=\mathbf{I}$), Brillouin's theorem ($F_{ia}^{\text{MO}}=0$), electron count ($\text{Tr}(\mathbf{P}\mathbf{S})=N_{\text{elec}}$), and McWeeny idempotency ($\mathbf{D}\mathbf{S}\mathbf{D}=\mathbf{D}$).
   - `compute_frontier_orbitals_and_koopmans_ip`: Extract $\varepsilon_{\text{HOMO}}$, $\varepsilon_{\text{LUMO}}$, the HOMO–LUMO gap, and Koopmans' vertical ionization potential $\text{IP} = -\varepsilon_{\text{HOMO}}$.
   - `compute_mulliken_lowdin_and_mayer`: Compute Mulliken partial charges $Q_A^{\text{Mulliken}}$, Löwdin partial charges $Q_A^{\text{Löwdin}}$, and Mayer bond orders $B_{AB}$ ([PySCF `scf.hf`](https://pyscf.org/_modules/pyscf/scf/hf.html)).
   - `compute_molecular_dipole_moment`: Evaluate $\boldsymbol{\mu} = \sum_A Z_A\mathbf{R}_A - \text{Tr}(\mathbf{P}\mathbf{r}_{\text{AO}})$ in atomic units and Debye.
6. **Part 6 — Step-by-Step Checkpoint & Verification Suite (`run_section4_checkpoints`)**.

In [ ]:
import numpy as np
from scipy.linalg import eigh
import matplotlib.pyplot as plt

np.set_printoptions(precision=5, suppress=True, linewidth=110)
HARTREE_TO_EV = 27.211386245988
AU_TO_DEBYE = 2.541746473

# Load exact STO-3G reference archives for offline testing in Part 6
heh_archive = np.load("heh_plus_sto3g_integrals.npz")
h2o_archive = np.load("h2o_sto3g_integrals.npz")
print("Loaded HeH+ and H2O STO-3G integral archives for checkpoint verification.")

## Part 1: Extracting Molecular Integrals with PySCF & Verifying ERI Symmetries (TODOs 1–2)

In [PySCF](https://pyscf.org/user/gto.html), building a `gto.Mole` object gives direct access to `libcint` integrals:
- `S = mol.intor("int1e_ovlp")`, `T = mol.intor("int1e_kin")`, `V = mol.intor("int1e_nuc")`
- `H_core = T + V`, `eri = mol.intor("int2e")`
- `with mol.with_common_orig((0.0, 0.0, 0.0)): ao_dip = mol.intor_symmetric("int1e_r", comp=3)`
- `e_nuc = float(mol.energy_nuc())`, `n_elec = int(mol.nelectron)`

Because real Gaussian basis functions satisfy $\chi_\mu(\mathbf{r})\chi_\nu(\mathbf{r}) = \chi_\nu(\mathbf{r})\chi_\mu(\mathbf{r})$ and the Coulomb operator $r_{12}^{-1}$ is symmetric in electrons 1 and 2, the 4-index ERI tensor $(\mu\nu|\lambda\sigma)$ obeys **8-fold permutational symmetry**:
$$(\mu\nu|\lambda\sigma) = (\nu\mu|\lambda\sigma) = (\mu\nu|\sigma\lambda) = (\lambda\sigma|\mu\nu)$$

In [ ]:
def extract_ao_integrals_from_pyscf(
    atom_spec: str = "O 0.0 0.0 0.0; H 0.0 -0.757 0.587; H 0.0 0.757 0.587",
    basis: str = "sto-3g",
    fallback_npz: str = "h2o_sto3g_integrals.npz",
) -> dict:
    '''
    TODO 1: Extract S, T, V, H_core, eri, ao_dip, e_nuc, n_elec, atom_charges, and atom_coords_bohr.
    If PySCF is installed, extract them directly from `pyscf.gto.M(atom=atom_spec, basis=basis, verbose=0)`;
    if `ImportError` occurs, load them from `fallback_npz` and compute `H_core = T + V`.
    '''
    # TODO 1: Try importing `from pyscf import gto`, build `mol`, and extract all required arrays.
    #         In the `except ImportError:` block, load `np.load(fallback_npz)` and form `H_core = T + V`.
    raise NotImplementedError("Complete TODO 1: extract_ao_integrals_from_pyscf")


def verify_eri_8fold_symmetry(eri: np.ndarray) -> float:
    '''
    TODO 2: Return the total Frobenius norm of the three generator symmetry residuals:
    ||(uv|ls) - (vu|ls)||_F + ||(uv|ls) - (uv|sl)||_F + ||(uv|ls) - (ls|uv)||_F.
    '''
    # TODO 2: Use np.transpose(eri, ...) to permute axes (1,0,2,3), (0,1,3,2), and (2,3,0,1).
    raise NotImplementedError("Complete TODO 2: verify_eri_8fold_symmetry")

## Part 2: Symmetric Orthogonalization ($\mathbf{X} = \mathbf{S}^{-1/2}$), Roothaan–Hall Step & Density Matrix (TODOs 3–5)

1. **Löwdin Symmetric Orthogonalization (`build_symmetric_orthogonalizer`)**:
   Diagonalize $\mathbf{S} = \mathbf{U}\mathbf{s}\mathbf{U}^\mathsf{T}$ with `scipy.linalg.eigh(S)` and construct:
   $$\mathbf{X} = \mathbf{S}^{-1/2} = \mathbf{U}\,\text{diag}(s_\alpha^{-1/2})\,\mathbf{U}^\mathsf{T}$$
2. **Single Roothaan–Hall Diagonalization Step (`solve_roothaan_hall_step`)**:
   Given any AO Fock matrix $\mathbf{F}$ and orthogonalizer $\mathbf{X}$:
   $$\mathbf{F}' = \mathbf{X}^\mathsf{T}\mathbf{F}\mathbf{X}, \qquad \mathbf{F}'\mathbf{C}' = \mathbf{C}'\boldsymbol{\varepsilon}, \qquad \mathbf{C} = \mathbf{X}\mathbf{C}'$$
3. **Unscaled Spatial Occupied Density Matrix (`compute_density_matrix`)**:
   Slice the first $n_{\text{occ}} = N_{\text{elec}}/2$ columns $\mathbf{C}_{\text{occ}} = \mathbf{C}_{:, :n_{\text{occ}}}$ and contract:
   $$D_{\mu\nu} = \sum_{i=0}^{n_{\text{occ}}-1} C_{\mu i}\,C_{\nu i}$$

In [ ]:
def build_symmetric_orthogonalizer(S: np.ndarray, tol: float = 1e-10) -> np.ndarray:
    '''
    TODO 3: Compute the symmetric orthogonalization matrix X = S^(-1/2) = U s^(-1/2) U^T.
    '''
    # TODO 3: Diagonalize S via eigh(S), filter eigenvalues > tol, and return U @ diag(s^(-1/2)) @ U.T.
    raise NotImplementedError("Complete TODO 3: build_symmetric_orthogonalizer")


def solve_roothaan_hall_step(F: np.ndarray, X: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    '''
    TODO 4: Transform F into the orthogonalized basis F' = X^T F X, diagonalize with eigh,
    back-transform C = X C', and return (eps, C).
    '''
    # TODO 4: Implement F' = X.T @ F @ X, diagonalize F', and return (eps, X @ C').
    raise NotImplementedError("Complete TODO 4: solve_roothaan_hall_step")


def compute_density_matrix(C: np.ndarray, n_occ: int) -> np.ndarray:
    '''
    TODO 5: Compute the unscaled spatial occupied density matrix D_{uv} = sum_{i=0}^{n_occ-1} C_{u,i} C_{v,i}.
    '''
    # TODO 5: Slice C_occ = C[:, :n_occ] and contract via np.einsum or matrix multiplication.
    raise NotImplementedError("Complete TODO 5: compute_density_matrix")

## Part 3: Forming Coulomb ($\mathbf{J}$), Exchange ($\mathbf{K}$), Fock ($\mathbf{F}$), and RHF Energies (TODOs 6–8)

1. **Coulomb and Exchange Contractions (`compute_coulomb_and_exchange_matrices`)**:
   $$J_{\mu\nu} = \sum_{\lambda\sigma} (\mu\nu|\lambda\sigma)\,D_{\lambda\sigma}, \qquad K_{\mu\nu} = \sum_{\lambda\sigma} (\mu\lambda|\nu\sigma)\,D_{\lambda\sigma}$$
2. **Fock Matrix (`build_fock_matrix`)**:
   $$\mathbf{F} = \mathbf{H}^{\text{core}} + 2\mathbf{J} - \mathbf{K}$$
3. **Electronic and Total RHF Energy (`compute_rhf_energies`)**:
   $$E_{\text{elec}} = \sum_{\mu\nu} D_{\mu\nu}\left(H_{\mu\nu}^{\text{core}} + F_{\mu\nu}\right), \qquad E_{\text{total}} = E_{\text{elec}} + E_{\text{nuc}}$$
   *(Think carefully: why is $E_{\text{elec}}$ equal to $\text{Tr}[\mathbf{D}(\mathbf{H}^{\text{core}} + \mathbf{F})]$ rather than $2\text{Tr}[\mathbf{D}\mathbf{F}]$?)*

In [ ]:
def compute_coulomb_and_exchange_matrices(
    eri: np.ndarray, D: np.ndarray
) -> tuple[np.ndarray, np.ndarray]:
    '''
    TODO 6: Contract the 4-index ERI tensor (uv|ls) with D_{ls} using np.einsum
    to return (J, K).
    '''
    # TODO 6: Compute J = einsum("uvls,ls->uv", eri, D) and K = einsum("ulvs,ls->uv", eri, D).
    raise NotImplementedError("Complete TODO 6: compute_coulomb_and_exchange_matrices")


def build_fock_matrix(H_core: np.ndarray, J: np.ndarray, K: np.ndarray) -> np.ndarray:
    '''
    TODO 7: Construct the closed-shell RHF Fock matrix F = H_core + 2*J - K.
    '''
    # TODO 7: Return H_core + 2.0 * J - K.
    raise NotImplementedError("Complete TODO 7: build_fock_matrix")


def compute_rhf_energies(
    D: np.ndarray, H_core: np.ndarray, F: np.ndarray, e_nuc: float
) -> tuple[float, float]:
    '''
    TODO 8: Compute and return (e_elec, e_total).
    '''
    # TODO 8: Contract D with (H_core + F) to obtain e_elec, and return (e_elec, e_elec + e_nuc).
    raise NotImplementedError("Complete TODO 8: compute_rhf_energies")

## Part 4: Pulay DIIS Extrapolation & Complete SCF Loop (TODOs 9–11)

1. **Orthonormalized Pulay Orbital Gradient (`compute_pulay_orbital_gradient`)**:
   $$\mathbf{e}_k = \mathbf{X}^\mathsf{T}\left(\mathbf{F}_k \mathbf{D}_k \mathbf{S} - \mathbf{S} \mathbf{D}_k \mathbf{F}_k\right)\mathbf{X}$$
2. **Pulay DIIS Linear System (`solve_diis_extrapolation`)**:
   Given a history of $m$ Fock matrices $\{\mathbf{F}_i\}_{i=1}^m$ and error matrices $\{\mathbf{e}_i\}_{i=1}^m$, construct the $(m+1)\times(m+1)$ augmented matrix $\mathbf{B}$ where $B_{ij} = \text{Tr}(\mathbf{e}_i^\mathsf{T}\mathbf{e}_j)$, $B_{i,m+1} = B_{m+1,i} = -1$, and $B_{m+1,m+1} = 0$, solve $\mathbf{B}\mathbf{c} = [0, \dots, 0, -1]^\mathsf{T}$, and return $\mathbf{F}_{\text{DIIS}} = \sum_{i=1}^m c_i \mathbf{F}_i$ ([Psi4NumPy RHF-DIIS](https://github.com/psi4/psi4numpy/blob/master/Tutorials/03_Hartree-Fock/3b_rhf-diis.ipynb)).
3. **Complete RHF SCF Driver (`run_rhf_scf_loop`)**:
   Combine TODOs 3–10 into the iterative SCF procedure starting from the core guess $\mathbf{F}^{(0)} = \mathbf{H}^{\text{core}}$. Once $|\Delta E_k| < e_{\text{tol}}$ and $\|\mathbf{e}_k\|_{\text{RMS}} < d_{\text{tol}}$, perform one final canonical diagonalization of the converged Fock matrix $\mathbf{F}$ so $\mathbf{C}^\mathsf{T}\mathbf{F}\mathbf{C} = \text{diag}(\boldsymbol{\varepsilon})$ holds to machine precision.

In [ ]:
def compute_pulay_orbital_gradient(
    F: np.ndarray, D: np.ndarray, S: np.ndarray, X: np.ndarray
) -> tuple[np.ndarray, float]:
    '''
    TODO 9: Compute the orthonormalized Pulay error matrix e = X^T (FDS - SDF) X
    and its root-mean-square norm rms_error = sqrt(mean(e^2)). Return (e, rms_error).
    '''
    # TODO 9: Compute fds = F @ D @ S, e = X.T @ (fds - fds.T) @ X, and rms_error.
    raise NotImplementedError("Complete TODO 9: compute_pulay_orbital_gradient")


def solve_diis_extrapolation(
    fock_list: list[np.ndarray], error_list: list[np.ndarray]
) -> np.ndarray:
    '''
    TODO 10: Solve the (m+1) x (m+1) Pulay DIIS linear system and return F_diis = sum_i c_i F_i.
    '''
    # TODO 10: Build the bordered B matrix with B_{ij} = einsum("ab,ab->", e_i, e_j),
    #          solve B @ coeffs = [0, ..., 0, -1], and return sum(c_i * F_i).
    raise NotImplementedError("Complete TODO 10: solve_diis_extrapolation")


def run_rhf_scf_loop(
    S: np.ndarray,
    T: np.ndarray,
    V: np.ndarray,
    eri: np.ndarray,
    e_nuc: float,
    n_elec: int,
    max_iter: int = 100,
    e_tol: float = 1e-10,
    d_tol: float = 1e-8,
    use_diis: bool = True,
    diis_max_space: int = 6,
) -> dict:
    '''
    TODO 11: Execute the complete RHF SCF cycle using TODOs 3-10.

    Returns
    -------
    dict with keys:
        'converged', 'e_total', 'e_elec', 'e_nuc', 'eps', 'C', 'D', 'F', 'J', 'K',
        'H_core', 'X', 'n_iterations', 'history'
    '''
    # TODO 11: Form H_core = T + V, X = S^(-1/2), and initial core guess (eps, C, D).
    #          In each iteration, build (J, K, F), evaluate (e_elec, e_total), compute (grad, rms_grad),
    #          check convergence, extrapolate F via DIIS if use_diis is True and iter >= 2,
    #          and update (eps, C, D) via solve_roothaan_hall_step.
    raise NotImplementedError("Complete TODO 11: run_rhf_scf_loop")

## Part 5: Checking Converged Energies, Wavefunction Invariants & Molecular Properties (TODOs 12–16)

With a converged RHF solution $(\mathbf{D}, \mathbf{C}, \mathbf{F}, \mathbf{J}, \mathbf{K}, \boldsymbol{\varepsilon})$ and spin-summed charge density matrix $\mathbf{P} = 2\mathbf{D}$, implement five post-SCF analysis functions:
1. **Energy Breakdown & Virial Ratio (`compute_energy_components_and_virial` — TODO 12)**:
   $$E_{\text{kin}} = \text{Tr}(\mathbf{P}\mathbf{T}), \quad E_{\text{ne}} = \text{Tr}(\mathbf{P}\mathbf{V}), \quad E_J = 2\text{Tr}(\mathbf{D}\mathbf{J}), \quad E_K = -\text{Tr}(\mathbf{D}\mathbf{K}), \quad \eta_{\text{virial}} = -\frac{E_{\text{ne}} + E_J + E_K + E_{\text{nuc}}}{E_{\text{kin}}}$$
2. **Mathematical Wavefunction Invariants (`verify_wavefunction_invariants` — TODO 13)**:
   Check MO orthonormality $\|\mathbf{C}^\mathsf{T}\mathbf{S}\mathbf{C} - \mathbf{I}\|_F$, Brillouin's theorem $\|\mathbf{F}_{\text{occ,virt}}^{\text{MO}}\|_F$ (where $\mathbf{F}^{\text{MO}} = \mathbf{C}^\mathsf{T}\mathbf{F}\mathbf{C}$), total electron count $\text{Tr}(\mathbf{P}\mathbf{S})$, and McWeeny idempotency $\|\mathbf{D}\mathbf{S}\mathbf{D} - \mathbf{D}\|_F$.
3. **Frontier Orbitals & Koopmans' Theorem (`compute_frontier_orbitals_and_koopmans_ip` — TODO 14)**:
   Extract $\varepsilon_{\text{HOMO}} = \varepsilon_{n_{\text{occ}}-1}$, $\varepsilon_{\text{LUMO}} = \varepsilon_{n_{\text{occ}}}$, $\Delta\varepsilon_{\text{gap}} = \varepsilon_{\text{LUMO}} - \varepsilon_{\text{HOMO}}$, and Koopmans' ionization potential $\text{IP}_{\text{eV}} = -27.211386\,\varepsilon_{\text{HOMO}}$.
4. **Mulliken, Löwdin & Mayer Population Analysis (`compute_mulliken_lowdin_and_mayer` — TODO 15)**:
   $$Q_A^{\text{Mulliken}} = Z_A - \sum_{\mu \in A} (\mathbf{P}\mathbf{S})_{\mu\mu}, \qquad Q_A^{\text{Löwdin}} = Z_A - \sum_{\mu \in A} (\mathbf{S}^{1/2}\mathbf{P}\mathbf{S}^{1/2})_{\mu\mu}, \qquad B_{AB} = \sum_{\mu \in A}\sum_{\nu \in B} (\mathbf{P}\mathbf{S})_{\mu\nu}(\mathbf{P}\mathbf{S})_{\nu\mu}$$
5. **Molecular Electric Dipole Moment (`compute_molecular_dipole_moment` — TODO 16)**:
   $$\boldsymbol{\mu}_{\text{a.u.}} = \sum_A Z_A\mathbf{R}_A - \sum_{\mu\nu} P_{\mu\nu}\langle\chi_\mu|\hat{\mathbf{r}}|\chi_\nu\rangle, \qquad \|\boldsymbol{\mu}\|_{\text{Debye}} = 2.541746473\,\|\boldsymbol{\mu}_{\text{a.u.}}\|_2$$

In [ ]:
def compute_energy_components_and_virial(
    D: np.ndarray, T: np.ndarray, V: np.ndarray, J: np.ndarray, K: np.ndarray, e_nuc: float
) -> dict[str, float]:
    '''
    TODO 12: Return a dict with keys 'e_kin', 'e_ne', 'e_1e', 'e_coul', 'e_exch',
    'e_2e', 'e_nuc', 'e_total', and 'virial_ratio'.
    '''
    # TODO 12: Evaluate each energy component from P = 2*D and D, and compute virial_ratio = -V_tot / E_kin.
    raise NotImplementedError("Complete TODO 12: compute_energy_components_and_virial")


def verify_wavefunction_invariants(
    C: np.ndarray, D: np.ndarray, F: np.ndarray, S: np.ndarray, eps: np.ndarray, n_occ: int
) -> dict[str, float]:
    '''
    TODO 13: Return a dict with keys 'mo_ortho_err', 'brillouin_err', 'fock_diag_err',
    'n_elec_trace', and 'idempotency_err'.
    '''
    # TODO 13: Compute ||C^T S C - I||_F, ||(C^T F C)[:n_occ, n_occ:]||_F,
    #          ||C^T F C - diag(eps)||_F, Tr(2 D S), and ||D S D - D||_F.
    raise NotImplementedError("Complete TODO 13: verify_wavefunction_invariants")


def compute_frontier_orbitals_and_koopmans_ip(
    eps: np.ndarray, n_occ: int
) -> dict[str, float]:
    '''
    TODO 14: Return a dict with keys 'eps_homo_eh', 'eps_lumo_eh', 'gap_eh', 'gap_ev', and 'koopmans_ip_ev'.
    '''
    # TODO 14: Extract HOMO (n_occ - 1) and LUMO (n_occ) energies and convert to eV using HARTREE_TO_EV.
    raise NotImplementedError("Complete TODO 14: compute_frontier_orbitals_and_koopmans_ip")


def compute_mulliken_lowdin_and_mayer(
    D: np.ndarray, S: np.ndarray, atom_charges: np.ndarray, ao_to_atom: list[int]
) -> dict[str, np.ndarray]:
    '''
    TODO 15: Compute Mulliken partial charges, Löwdin partial charges, and Mayer bond order matrix.
    Return a dict with keys 'q_mulliken', 'q_lowdin', 'mayer_bond_order', 'ps_diag', 'lowdin_diag'.
    '''
    # TODO 15: Form P = 2*D, PS = P @ S, S^(1/2) via eigh(S), and P_lowdin = S^(1/2) P S^(1/2).
    #          Accumulate atomic partial charges Q_A and Mayer bond orders B_{AB}.
    raise NotImplementedError("Complete TODO 15: compute_mulliken_lowdin_and_mayer")


def compute_molecular_dipole_moment(
    D: np.ndarray,
    ao_dip: np.ndarray,
    atom_charges: np.ndarray,
    atom_coords_bohr: np.ndarray,
) -> tuple[np.ndarray, float]:
    '''
    TODO 16: Compute the molecular electric dipole moment vector in atomic units (e * Bohr)
    and its magnitude in Debye. Return (dip_vec_au, dip_norm_debye).
    '''
    # TODO 16: Compute nuc_dip = sum_A Z_A R_A and elec_dip = einsum("xuv,vu->x", ao_dip, 2*D),
    #          then return (nuc_dip - elec_dip, ||nuc_dip - elec_dip||_2 * AU_TO_DEBYE).
    raise NotImplementedError("Complete TODO 16: compute_molecular_dipole_moment")

## Part 6: Step-by-Step Checkpoint & Verification Suite

Run `run_section4_checkpoints()` below as you implement each `# TODO` function. Every function is tested individually against exact `STO-3G` benchmarks on $\text{HeH}^+$ and $\text{H}_2\text{O}$:

| Target Property ($\text{H}_2\text{O}$ / `STO-3G`) | Exact Benchmark Value |
| :--- | :--- |
| **Total RHF Energy ($E_{\text{RHF}}$)** | $-74.9630620833\ E_{\text{h}}$ (8 iterations with DIIS vs. 18 without DIIS) |
| **Energy Components ($E_{\text{kin}}, E_{\text{ne}}, E_J, E_K$)** | $+74.58719122, -196.94594772, +47.31151149, -9.10407614\ E_{\text{h}}$ |
| **Virial Ratio ($-\langle\hat{V}\rangle/\langle\hat{T}\rangle$)** | $2.005039$ |
| **HOMO / LUMO / Koopmans' IP** | $\varepsilon_{\text{HOMO}} = -0.391274\ E_{\text{h}}$, $\varepsilon_{\text{LUMO}} = +0.605136\ E_{\text{h}}$, $\text{IP} = 10.65\text{ eV}$ |
| **Mulliken vs. Löwdin Charges on Oxygen** | $Q_{\text{O}}^{\text{Mulliken}} = -0.36551\,e$, $Q_{\text{O}}^{\text{Löwdin}} = -0.25287\,e$ |
| **Mayer Bond Order $B_{\text{O}-\text{H}}$ & Dipole $\|\boldsymbol{\mu}\|$** | $B_{\text{O}-\text{H}} = 0.9542$, $\|\boldsymbol{\mu}\| = 1.7255\text{ Debye}$ |

In [2]:
def run_section4_checkpoints() -> None:
    '''Step-by-step checkpoint verifier for all 16 #TODO functions in Section 4.'''
    passed = 0
    total_todos = 16

    try:
        # --- PART 1 CHECKS (TODOs 1-2) ---
        mol_data = extract_ao_integrals_from_pyscf()
        S, T, V, H_core, eri = mol_data["S"], mol_data["T"], mol_data["V"], mol_data["H_core"], mol_data["eri"]
        e_nuc, n_elec = float(mol_data["e_nuc"]), int(mol_data["n_elec"])
        assert S.shape == (7, 7) and eri.shape == (7, 7, 7, 7) and np.allclose(H_core, T + V)
        passed += 1
        print("[PASS] TODO 1 : extract_ao_integrals_from_pyscf")

        sym_err = verify_eri_8fold_symmetry(eri)
        assert sym_err < 1e-13
        passed += 1
        print(f"[PASS] TODO 2 : verify_eri_8fold_symmetry (error = {sym_err:.2e})")

        # --- PART 2 CHECKS (TODOs 3-5) ---
        X = build_symmetric_orthogonalizer(S)
        assert X.shape == (7, 7) and np.linalg.norm(X.T @ S @ X - np.eye(7)) < 1e-13
        passed += 1
        print("[PASS] TODO 3 : build_symmetric_orthogonalizer")

        eps_0, C_0 = solve_roothaan_hall_step(H_core, X)
        assert eps_0.shape == (7,) and C_0.shape == (7, 7) and np.linalg.norm(C_0.T @ S @ C_0 - np.eye(7)) < 1e-13
        passed += 1
        print("[PASS] TODO 4 : solve_roothaan_hall_step")

        D_0 = compute_density_matrix(C_0, n_occ=5)
        assert D_0.shape == (7, 7) and np.isclose(2.0 * np.trace(D_0 @ S), 10.0, atol=1e-12)
        passed += 1
        print("[PASS] TODO 5 : compute_density_matrix")

        # --- PART 3 CHECKS (TODOs 6-8) ---
        J_0, K_0 = compute_coulomb_and_exchange_matrices(eri, D_0)
        assert J_0.shape == (7, 7) and K_0.shape == (7, 7) and np.isclose(J_0[0, 0], 9.387679, atol=1e-5)
        passed += 1
        print("[PASS] TODO 6 : compute_coulomb_and_exchange_matrices")

        F_0 = build_fock_matrix(H_core, J_0, K_0)
        assert np.allclose(F_0, H_core + 2.0 * J_0 - K_0)
        passed += 1
        print("[PASS] TODO 7 : build_fock_matrix")

        e_elec_1, e_tot_1 = compute_rhf_energies(D_0, H_core, F_0, e_nuc)
        assert np.isclose(e_tot_1, -73.23270896, atol=1e-6)
        passed += 1
        print(f"[PASS] TODO 8 : compute_rhf_energies (Iter 1 E_tot = {e_tot_1:.8f} Eh)")

        # --- PART 4 CHECKS (TODOs 9-11) ---
        grad_0, rms_0 = compute_pulay_orbital_gradient(F_0, D_0, S, X)
        assert grad_0.shape == (7, 7) and np.isclose(np.linalg.norm(grad_0 + grad_0.T), 0.0, atol=1e-13)
        assert np.isclose(rms_0, 0.148914, atol=1e-5)
        passed += 1
        print(f"[PASS] TODO 9 : compute_pulay_orbital_gradient (Iter 1 RMS grad = {rms_0:.4e})")

        F_extrap = solve_diis_extrapolation([H_core, F_0], [grad_0 * 2.0, grad_0])
        assert F_extrap.shape == (7, 7)
        passed += 1
        print("[PASS] TODO 10: solve_diis_extrapolation")

        res_diis = run_rhf_scf_loop(S, T, V, eri, e_nuc, n_elec, use_diis=True)
        res_nodiis = run_rhf_scf_loop(S, T, V, eri, e_nuc, n_elec, use_diis=False)
        assert res_diis["converged"] and np.isclose(res_diis["e_total"], -74.9630620833, atol=1e-8)
        assert res_diis["n_iterations"] < res_nodiis["n_iterations"]
        passed += 1
        print(f"[PASS] TODO 11: run_rhf_scf_loop (E_RHF = {res_diis['e_total']:.10f} Eh in {res_diis['n_iterations']} DIIS iters)")

        # --- PART 5 CHECKS (TODOs 12-16) ---
        e_comp = compute_energy_components_and_virial(res_diis["D"], T, V, res_diis["J"], res_diis["K"], e_nuc)
        assert np.isclose(e_comp["virial_ratio"], 2.005039, atol=1e-5)
        passed += 1
        print(f"[PASS] TODO 12: compute_energy_components_and_virial (-<V>/<T> = {e_comp['virial_ratio']:.6f})")

        inv = verify_wavefunction_invariants(res_diis["C"], res_diis["D"], res_diis["F"], S, res_diis["eps"], n_occ=5)
        assert inv["mo_ortho_err"] < 1e-12 and inv["brillouin_err"] < 1e-9 and inv["idempotency_err"] < 1e-12
        assert np.isclose(inv["n_elec_trace"], 10.0, atol=1e-12)
        passed += 1
        print("[PASS] TODO 13: verify_wavefunction_invariants (all errors < 1e-12)")

        front = compute_frontier_orbitals_and_koopmans_ip(res_diis["eps"], n_occ=5)
        assert np.isclose(front["koopmans_ip_ev"], 10.6471, atol=1e-3)
        passed += 1
        print(f"[PASS] TODO 14: compute_frontier_orbitals_and_koopmans_ip (IP = {front['koopmans_ip_ev']:.2f} eV)")

        pop = compute_mulliken_lowdin_and_mayer(
            res_diis["D"], S, h2o_archive["atom_charges"], [0, 0, 0, 0, 0, 1, 2]
        )
        assert np.isclose(pop["q_mulliken"][0], -0.36551, atol=1e-4)
        assert np.isclose(pop["q_lowdin"][0], -0.25287, atol=1e-4)
        assert np.isclose(pop["mayer_bond_order"][0, 1], 0.9542, atol=1e-3)
        passed += 1
        print(f"[PASS] TODO 15: compute_mulliken_lowdin_and_mayer (q_O_Mulliken = {pop['q_mulliken'][0]:+.4f} e)")

        dip_vec, dip_debye = compute_molecular_dipole_moment(
            res_diis["D"], h2o_archive["ao_dip"], h2o_archive["atom_charges"], h2o_archive["atom_coords_bohr"]
        )
        assert np.isclose(dip_debye, 1.7255, atol=1e-3)
        passed += 1
        print(f"[PASS] TODO 16: compute_molecular_dipole_moment (||mu|| = {dip_debye:.4f} Debye)")
        print(f"\nCONGRATULATIONS! All {passed}/{total_todos} Section 4 Hartree-Fock checkpoints passed!")

    except NotImplementedError as exc:
        print(f"\n[PROGRESS: {passed}/{total_todos} checkpoints passed] Next up -> {exc}")
    except AssertionError as exc:
        print(f"\n[ASSERTION FAILED after {passed}/{total_todos} checkpoints] -> {exc}")


# Uncomment and run as you implement each TODO:
# run_section4_checkpoints()
print("Checkpoint suite loaded. Implement TODOs 1-16 above and call `run_section4_checkpoints()` to test your code!")

Checkpoint suite loaded. Implement TODOs 1-16 above and call `run_section4_checkpoints()` to test your code!
